# Compare cached features: configurable entry point
Five repeated 80:20 **image-level** holdouts on cached features. No scaling, tuning, stratification, patient grouping or extractor refitting is silently introduced. New results are saved apart from the historical article summaries.

For raw images, start with notebooks 00–02. For a feature cache, run this notebook directly.

In [ ]:
from pathlib import Path
import sys, zipfile, subprocess
PROJECT_ROOT = Path('/content/DMAE_Colab_Repository')
if not PROJECT_ROOT.exists():
    local = Path.cwd()
    if (local / 'src' / 'amd_oct').exists():
        PROJECT_ROOT = local
    else:
        from google.colab import files
        print('Upload DMAE_Colab_Repository.zip (code package, not the image database).')
        uploaded = files.upload()
        package = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if package is None:
            raise ValueError('Select the code ZIP package.')
        destination = Path('/content').resolve()
        with zipfile.ZipFile(package) as archive:
            for member in archive.infolist():
                resolved = (destination / member.filename).resolve()
                if not resolved.is_relative_to(destination):
                    raise ValueError('Unsafe archive path.')
            archive.extractall(destination)
if not (PROJECT_ROOT / 'src' / 'amd_oct').is_dir():
    raise FileNotFoundError('Set PROJECT_ROOT to the extracted code-package directory.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Code root:', PROJECT_ROOT)


In [ ]:
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/AMD_OCT') if USE_GOOGLE_DRIVE else Path('/content/AMD_OCT')
DATASET_DIR = BASE_DIR / 'data'  # Contains Normal, iORA, cORA, iRORA, cRORA folders.
CACHE_DIR = BASE_DIR / 'features'
MODEL_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
for directory in (CACHE_DIR, MODEL_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# Classical classifier: TensorFlow is not required.

In [ ]:
from datetime import datetime, timezone
from amd_oct.data import CACHE_CLASS_ORDER
from amd_oct.experiment import run_experiment
EXTRACTOR = 'custom_cnn'
CLASSIFIER = 'svm'
CACHE_PATH = CACHE_DIR / {'custom_cnn':'DB_CNNpropia.npz','vgg16':'DB_VGG16.npz',
    'resnet50':'DB_ResNet50.npz','efficientnet':'DB_EfficientNet.npz'}[EXTRACTOR]
# Set CLASS_NAMES=None for new caches containing their own class_names metadata.
# This explicit order is verified for the supplied legacy CNN/VGG feature caches.
# Verify it before using a different legacy cache, especially an exploratory backbone.
CLASS_NAMES = CACHE_CLASS_ORDER
REPETITIONS = 5
SPLIT_SEED = None  # Historical unseeded image-level protocol. An integer creates a new seeded analysis.
RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = RESULTS_DIR / f'{EXTRACTOR}_{CLASSIFIER}_{RUN_NAME}'


In [ ]:
runs, summary = run_experiment(CACHE_PATH, OUTPUT_DIR, EXTRACTOR, CLASSIFIER,
    repetitions=REPETITIONS, split_seed=SPLIT_SEED, class_names=CLASS_NAMES)
display(runs)
display(summary)
print('Results:', OUTPUT_DIR)
